# IMDB Sentiment - RNN Family Comparison

Notebook này chạy 4 kiến trúc thuộc họ RNN trên một dataset chuỗi để so sánh.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras import layers

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "dataset"
RESULTS_DIR = BASE_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)
sns.set_theme(style="whitegrid")
tf.random.set_seed(42)
np.random.seed(42)


In [ ]:
df = pd.read_csv(DATA_DIR / "IMDB Dataset.csv").sample(10000, random_state=42)
texts = df["review"].to_numpy()
labels = df["sentiment"].map({"negative": 0, "positive": 1}).to_numpy()

tokenizer = keras.preprocessing.text.Tokenizer(num_words=10000, oov_token="<unk>")
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)
X = keras.preprocessing.sequence.pad_sequences(sequences, maxlen=200)
y = labels
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
vocab_size = 10000
num_classes = 2


In [ ]:
def build_simple_rnn(vocab_size, num_classes):
    model = keras.Sequential([
        layers.Embedding(vocab_size, 128),
        layers.SimpleRNN(64),
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

def build_birnn(vocab_size, num_classes):
    model = keras.Sequential([
        layers.Embedding(vocab_size, 128),
        layers.Bidirectional(layers.SimpleRNN(64)),
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

def build_gru(vocab_size, num_classes):
    model = keras.Sequential([
        layers.Embedding(vocab_size, 128),
        layers.GRU(64),
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

def build_deep_rnn(vocab_size, num_classes):
    model = keras.Sequential([
        layers.Embedding(vocab_size, 128),
        layers.SimpleRNN(64, return_sequences=True),
        layers.SimpleRNN(64),
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

builders = {
    "SimpleRNN": build_simple_rnn,
    "BiRNN": build_birnn,
    "GRU": build_gru,
    "DeepRNN": build_deep_rnn,
}

results = []
for name, builder in builders.items():
    model = builder(vocab_size, num_classes)
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    model.fit(x_train, y_train, validation_data=(x_test, y_test), epochs=3, batch_size=128, verbose=0)
    loss, acc = model.evaluate(x_test, y_test, verbose=0)
    results.append({"model": name, "test_accuracy": acc, "test_loss": loss})

results_df = pd.DataFrame(results).sort_values("test_accuracy", ascending=False)
results_df

sns.barplot(data=results_df, x="test_accuracy", y="model", palette="magma")
plt.title("RNN-family Comparison")
plt.show()
